# 🛰️ ASTRA-SIGHT: Deep Neural Earth Observation (EO) Intelligence
**Objective:** 99% Precision on Satellite Imagery

This notebook implements the **Pro Max [Text-Submission Edition]** pipeline.
- **Smart Path Detection**: Automatically finds Kaggle datasets.
- **EfficientNet-B4**: Advanced resolution for spectral clarity.
- **Text-Label Submission**: Generates `FINAL.csv` with class names (e.g., 'River') instead of numbers.

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from torch.amp import GradScaler, autocast

# --- 1. SYSTEM CONFIGURATION ---
CONFIG = {
    'seed': 42,
    'img_size': 384,      
    'batch_size': 16,     
    'grad_accum': 2,      
    'epochs': 30,      
    'lr': 1e-4,        
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

def find_paths():
    """Smart Discovery Engine: Finds CSV and Images anywhere on Kaggle"""
    csv_matches = glob.glob('/kaggle/input/**/[Tt][Rr][Aa][Ii][Nn].csv', recursive=True)
    if not csv_matches: csv_matches = glob.glob('**/[Tt][Rr][Aa][Ii][Nn].csv', recursive=True)
    
    if csv_matches:
        train_csv = csv_matches[0]
        base_path = os.path.dirname(train_csv)
        # Find Test CSV
        test_matches = glob.glob(f"{base_path}/**/[Tt][Ee][Ss][Tt].csv", recursive=True)
        test_csv = test_matches[0] if test_matches else os.path.join(base_path, 'TEST.csv')
        return base_path, train_csv, test_csv
    return "./", "TRAIN.csv", "TEST.csv"

BASE_PATH, TRAIN_CSV, TEST_CSV = find_paths()
print(f"🚀 ASTRA-SIGHT: Data Root Found at {BASE_PATH}")

# Seeds
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

In [ ]:
# --- 2. DATA ENGINE ---
class ASTRADataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False, label_map=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = label_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        p_paths = [os.path.join(self.root_dir, img_name), 
                   os.path.join(self.root_dir, 'train_images', img_name),
                   os.path.join(self.root_dir, 'test_images', img_name),
                   os.path.join(self.root_dir, 'images', img_name)]
        
        image = None
        for p in p_paths:
            if os.path.exists(p):
                image = Image.open(p).convert('RGB')
                break
        if image is None: image = Image.new('RGB', (CONFIG['img_size'], CONFIG['img_size']))
        
        if self.transform: image = self.transform(image)
        if self.is_test: return image
        
        # LABEL HANDLING
        label_raw = self.df.iloc[idx, 1]
        if self.label_map:
            return image, self.label_map[label_raw]
        return image, int(label_raw)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['img_size'], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# --- 3. MODEL BUILDER ---
def build_model(num_classes):
    model = models.efficientnet_b4(weights='DEFAULT')
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(CONFIG['device'])

# --- 4. TRAINING ENGINE ---
def run_astra_training():
    # 1. Prepare DF and Label Map
    df = pd.read_csv(TRAIN_CSV)
    
    # Auto-detect if labels are strings
    unique_labels = sorted(df.iloc[:, 1].unique())
    num_classes = len(unique_labels)
    label_map = {name: i for i, name in enumerate(unique_labels)}
    inv_label_map = {i: name for name, i in label_map.items()}
    print(f"🏷️ Mapping {num_classes} classes: {label_map}")

    # 2. Split
    train_df, val_df = random_split(df, [int(len(df)*0.85), len(df)-int(len(df)*0.85)])
    
    train_ds = ASTRADataset(df.iloc[train_df.indices], BASE_PATH, train_tf, label_map=label_map)
    val_ds = ASTRADataset(df.iloc[val_df.indices], BASE_PATH, val_tf, label_map=label_map)
    
    loader_args = {'batch_size': CONFIG['batch_size'], 'num_workers': 2, 'pin_memory': True}
    t_loader = DataLoader(train_ds, shuffle=True, **loader_args)
    v_loader = DataLoader(val_ds, shuffle=False, **loader_args)

    # 3. Training Setup
    model = build_model(num_classes)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler('cuda')

    best_f1 = 0
    for epoch in range(CONFIG['epochs']):
        model.train()
        pbar = tqdm(t_loader, desc=f"Epoch {epoch+1}")
        for i, (imgs, lbls) in enumerate(pbar):
            imgs, lbls = imgs.to(CONFIG['device']), lbls.to(CONFIG['device'])
            with autocast('cuda'):
                outs = model(imgs)
                loss = criterion(outs, lbls) / CONFIG['grad_accum']
            scaler.scale(loss).backward()
            if (i+1) % CONFIG['grad_accum'] == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            pbar.set_postfix({'loss': f"{loss.item()*CONFIG['grad_accum']:.4f}"})

        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for imgs, lbls in v_loader:
                outs = model(imgs.to(CONFIG['device']))
                vp.extend(torch.max(outs, 1)[1].cpu().numpy())
                vl.extend(lbls.numpy())
        
        f1 = f1_score(vl, vp, average='macro')
        print(f"⭐ Epoch {epoch+1} F1: {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            # Save both model and map
            torch.save({'model': model.state_dict(), 'inv_map': inv_label_map}, 'astra_best.pth')
            print("🔥 New Best Model Saved!")
        scheduler.step()
    return model, inv_label_map

trained_model, final_inv_map = run_astra_training()

In [ ]:
# --- 5. INFERENCE (TEXT LABEL OUTPUT) ---
def generate_submission(model, inv_label_map):
    print("🏁 Synthesizing FINAL.csv with Text Labels...")
    checkpoint = torch.load('astra_best.pth')
    model.load_state_dict(checkpoint['model'])
    inv_map = checkpoint['inv_map']
    model.eval()
    
    test_df = pd.read_csv(TEST_CSV)
    test_ds = ASTRADataset(test_df, BASE_PATH, val_tf, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False)
    
    preds_int = []
    with torch.no_grad():
        for imgs in tqdm(test_loader, desc="Inference"):
            outs = model(imgs.to(CONFIG['device']))
            preds_int.extend(torch.max(outs, 1)[1].cpu().numpy())
            
    # Map back to strings (e.g., 0 -> "River")
    preds_text = [inv_map[p] for p in preds_int]
            
    pd.DataFrame({'IMAGE': test_df.iloc[:, 0], 'LABEL': preds_text}).to_csv('FINAL.csv', index=False)
    print("✅ MISSION COMPLETE: FINAL.csv created with Text Labels.")

generate_submission(trained_model, final_inv_map)